# 04. Composite Grid Risk Engine & Operational Dispatch
**Power Grid AI - Risk Engine**

### Objectives:
- Synthesize unsupervised anomaly intensity, predicted operational stress, and DGA chemistry severity.
- Compute the **Composite Grid Risk Index** (0 to 100).
- Classify transformers into risk tiers (`LOW`, `MEDIUM`, `HIGH`, `CRITICAL`).
- Deliver operational decision matrices and automated mitigation actions for dispatchers.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load final risk outputs
risk_path = os.path.join('..', 'outputs', 'risk_results.csv')
df_risk = pd.read_csv(risk_path)
print(f"Loaded Risk Assessment Records: {len(df_risk)}")
df_risk.head(10)


### Fleet-Wide Risk Tier Distribution

In [ ]:
tier_counts = df_risk['risk_tier'].value_counts()
colors = {'LOW': '#2ecc71', 'MEDIUM': '#f39c12', 'HIGH': '#e67e22', 'CRITICAL': '#e74c3c'}
tier_colors = [colors.get(t, '#95a5a6') for t in tier_counts.index]

plt.figure(figsize=(7, 5))
plt.bar(tier_counts.index, tier_counts.values, color=tier_colors, edgecolor='black')
plt.title('Substation Transformer Fleet - Risk Tier Breakdown', fontsize=13, fontweight='bold')
plt.xlabel('Risk Tier')
plt.ylabel('Telemetry Intervals')
for i, v in enumerate(tier_counts.values):
    plt.text(i, v + 15, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


### Operational Decision Space: Anomaly Intensity vs Predicted Stress

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_risk,
    x='predicted_stress_index',
    y='anomaly_intensity',
    hue='risk_tier',
    palette=colors,
    s=50,
    alpha=0.8
)
plt.axvline(x=60, color='gray', linestyle=':', label='Stress Warning Threshold')
plt.axhline(y=60, color='gray', linestyle='--', label='Anomaly Warning Threshold')
plt.title('Multi-Factor Operational Risk Decision Boundary', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Operational Stress (0-100)')
plt.ylabel('Unsupervised Anomaly Intensity (0-100)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


### High & Critical Priority Incidents with Recommended Mitigation

In [ ]:
priority_incidents = df_risk[df_risk['risk_tier'].isin(['HIGH', 'CRITICAL'])].copy()
priority_incidents = priority_incidents.sort_values(by='composite_risk_score', ascending=False)

display(priority_incidents[[
    'timestamp', 'transformer_id', 'composite_risk_score',
    'risk_tier', 'recommended_action'
]].head(15))
